In [5]:
from datetime import datetime, timedelta

def read_time_stamp(file):
    start_time_str = None
    time_points = []

    with open(file, 'r') as f:
        for line in f:
            line = line.strip()

            # Extract start time
            if line.startswith('InitialTime'):
                start_time_str = line.split(':', 1)[1].strip().strip('/')

            # Extract timestamp from BufferLog
            elif line.startswith('BufferLog'):
                # Example:
                # BufferLog: 783, 57, 16, 100
                # Take the first value: 783
                buffer_values = line.split(':', 1)[1].strip()
                timestamp_ms = int(buffer_values.split(',')[0].strip())
                time_points.append(timestamp_ms)

    if start_time_str is None:
        raise ValueError("InitialTime not found in file.")

    # Convert the start time string into a datetime object
    start_time = datetime.strptime(start_time_str, "%d-%b-%y/%H-%M-%S-%f")

    # Calculate frame timestamps
    frame_data = [
        start_time + timedelta(milliseconds=nt)
        for nt in time_points
    ]

    return frame_data


In [ ]:
import os
import numpy as np

timestamp_dir = "time_stamps" # you can change this to the actual directory where your timestamp files are located
files = sorted([f for f in os.listdir(timestamp_dir) if f.endswith(".txt")])

for fname in files:
    fpath = os.path.join(timestamp_dir, fname)
    frame_data = read_time_stamp(fpath)
    
    # Calculate intervals in milliseconds
    intervals_ms = [
        (frame_data[i+1] - frame_data[i]).total_seconds() * 1000
        for i in range(len(frame_data) - 1)
    ]
    
    mean_interval = np.mean(intervals_ms)
    frame_rate = 1000 / mean_interval
    
    print(f"{fname}: {len(frame_data)} frames, mean interval = {mean_interval:.2f} ms, frame rate = {frame_rate:.2f} fps")


group1.txt: 1869 frames, mean interval = 80.34 ms, frame rate = 12.45 fps
group2.txt: 1876 frames, mean interval = 80.00 ms, frame rate = 12.50 fps
group3.txt: 1869 frames, mean interval = 80.30 ms, frame rate = 12.45 fps
group4.txt: 1877 frames, mean interval = 79.96 ms, frame rate = 12.51 fps
group5.txt: 1870 frames, mean interval = 80.27 ms, frame rate = 12.46 fps
